# 전처리 코드 통합본 — 2026-09-11

> **리뷰 전용 노트북.** 여기서 직접 실행하지 않는다. 각 셀은 `/workspace`의 원본 `.py` 파일 내용을
> 그대로 옮긴 것이고, 원본 파일이 정본이다(이 노트북을 고쳐도 실제 파이프라인은 안 바뀐다).
>
> **범위**: "9월 10일까지 전처리 작업한 코드"에 해당하는 것만 모았다 — 원본 CSV를 피처·라벨·split으로
> 만드는 본체(`preprocess_9class.py`, 9/2)와 그 재사용 로더·언더샘플·블록 후처리 모듈(`prep9/`), 그리고
> 09-10에 추가된 "학습에 쓸 표본을 고르는" 두 스크립트(`make_cluster_indices.py`, `make_ratio_indices.py`).
> 학습(`train9/lgb_ooc.py` 등)·블록 묶기 실행(`build_blocks_9class.py`)은 이미 다른 노트북(다중분류 파이프라인
> 통합본)에 있어 여기서는 뺐다.

## 구성

| # | 파일 | 역할 | 수정일 |
|---|---|---|---|
| 1 | `preprocess_9class.py` | 전처리 본체 — 원본 CSV → 81열 피처·9-class 라벨·split·언더샘플 인덱스·오라클 블록 | 09-02 |
| 2 | `prep9/nbcells.py` | 08-25 검증 통과 노트북(`preprocess_multiclass.ipynb`)의 정의 셀을 sha1 게이트로 재사용하는 로더 | 09-07 |
| 3 | `prep9/undersample.py` | k-means 층화 언더샘플링 구현(08-26 회의 결정 3) | 08-26 |
| 4 | `prep9/postprocess_blocks.py` | 거래별 예측을 "사건(블록)" 단위로 묶는 후처리 — 운영·평가 공용 | 09-02 |
| 5 | `make_cluster_indices.py` | HI-Large용 cluster/CSSMC 언더샘플 인덱스 생성(09-10, 청크 처리로 메모리 문제 해결) | 09-10 |
| 6 | `make_ratio_indices.py` | 기존 r10~r300과 같은 난수 흐름으로 r350~r500 확장(09-10) | 09-10 |

**부록에만 요약**: `prep9/lowmem.py`(563줄, cgroup 27GiB 상한 대응 메모리 절약판 — 함수 시그니처는 원본과 동일, 가장 마지막 섹션 참고)


## 1. 전처리 본체 — `preprocess_9class.py`

HI-Small/HI-Large 원본 CSV → 2단계 계층 파이프라인(1차 9-class 다중분류용 피처·라벨, 2차 패턴외 이진 대상 분리).
81열 피처 생성, 꼬리 절단(화두 17), 시간순 분할, 클러스터 기반 언더샘플링(08-26 회의 결정 3), 오라클 블록(정답으로 만든 상한선) 산출.

원본 경로: `preprocess_9class.py`

In [ ]:
#!/usr/bin/env python3
"""HI-Small 전처리 — 2단계 계층 파이프라인(1차 9-class · 2차 패턴외 이진)

근거: 2026-08-26 팀 회의. 이 스크립트는 회의에서 결정된 것만 구현하고,
회의가 정하지 않은 값은 임의로 고르지 않고 `assumptions` 에 이름을 붙여 남긴다.

  회의 결정 1  1차 = 8종 패턴 다중분류, 2차 = '패턴 외' 대상 이진분류  -> §A §C
  회의 결정 1  1차 다중분류는 9-Class                                  -> §A
  회의 결정 2  패턴 적발 = 패턴 블록 알림 / 패턴 외 적발 = 단건 알림    -> §F
  회의 결정 2  거래별 점수만 나오므로 묶어주는 후처리가 필수            -> §F
  회의 결정 3  Bipartite vs Stack 혼동 원시 데이터 심층 분석            -> §G
  회의 결정 3  1~10일 구간과 18일 전체 구간을 각각 처리                 -> --basis
  회의 결정 3  클러스터 기반 언더샘플링으로 정상 비율 조정              -> §E

피처·정제·분할은 새로 만들지 않는다. 2026-08-25 산출물
`preprocess_multiclass.ipynb`(팀 EDA 수치 5개 재현 검증 통과)의 정의 셀을 그대로 싣고,
회의가 바꾼 **라벨 구성과 하위 산출물**만 다시 짠다(§A 이후).
"""
from __future__ import annotations

import argparse
import gc
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, '/workspace')
from prep9 import nbcells, postprocess_blocks as pb, undersample as us, lowmem as lm   # noqa: E402

SEED = 42
# 2026-09-02: 컨테이너 RAM 상한이 cgroup 27.3 GiB 라(호스트 503 GB 아님) HI-Large 는 원래 함수로 OOM 이다.
# --lowmem 이면 노트북 원본을 건드리지 않고 prep9/lowmem.py 의 메모리 절약판 함수로 갈아끼운다(값 동일).
LOWMEM = False
LOWMEM_SCRATCH = Path('/workspace/processed_9class/_lowmem_scratch')
SPLIT_TAGS = ('tr', 'va', 'te')
CLASS8 = 8                                  # 9번째 클래스 = 패턴 외(정상 + 패턴외 세탁)
# 2026-09-02: 언더샘플 방식. 'cluster' = 원래 동작(MiniBatchKMeans 층화 + 무작위 4벌씩).
# 'random' = 무작위 4벌만(피처 행렬을 RAM 에 안 올림 — HI-Large 용). 'none' = 생략.
# 근거: 08-27 선택 결과가 full(언더샘플 없음)이었고, HI-Large 는 train 정상 1억 행이라
# 클러스터링이 몇 시간을 먹는데 쓰이는 곳이 없다. main() 이 --undersample 로 덮어쓴다.
UNDERSAMPLE_MODE = 'cluster'
BASES = {
    # 회의: "1~10일 구간과 18일 전체 구간 데이터의 패턴 분포 차이를 고려한 모델 실험"
    # 아래 두 키는 HI-Small 의 기간(10일/18일)을 이름에 박은 것이라 다른 세트에 쓰면 거짓말이 된다.
    # 기존 산출물 경로(processed_9class/HI-Small_10day)를 유지하려고 그대로 둔다.
    '10day': dict(tail_min_frac=0.05, note='공식 주 기간만(화두 17). 꼬리 8일 제외', only='HI-Small'),
    '18day': dict(tail_min_frac=0.00, note='꼬리 포함 전체 기간', only='HI-Small'),
    # 세트 무관 이름. 절단 기준은 '일별 거래량이 중앙값의 5% 미만인 꼬리'라는 상대 규칙이라
    # 기간 길이와 무관하게 그대로 성립한다(HI-Medium 16일, HI-Large 97일 + 꼬리).
    'main': dict(tail_min_frac=0.05, note='공식 주 기간만(화두 17). 저조 꼬리 제외'),
    'full': dict(tail_min_frac=0.00, note='꼬리 포함 전체 기간'),
}
# 데이터셋별 기본 basis — HI-Small 은 기존 이름을 유지해 산출물 경로가 바뀌지 않게 한다.
DEFAULT_BASIS = {'HI-Small': ['10day', '18day']}
DEFAULT_BASIS_OTHER = ['main', 'full']
# 팀 EDA 대조 상수는 HI-Small 실측값이다. 다른 세트에서는 대조 자체를 건너뛴다.
EDA_EXPECTED = {'HI-Small': {'label1_total': 5177, 'in_pattern': 3209, 'out_of_pattern': 1968}}
UNDERSAMPLE_RATIOS = (10, 30, 100, 300)     # 정상:패턴 목표 비율
KMEANS_KS = (64, 256, 1024)
BLOCK_WINDOWS = (60, 360, 720, 1440, 2160, 4320, 10080, 25920)   # 분


# ─────────────────────────────────────────────────────────────────────────────
# §A. 9-class 라벨
# ─────────────────────────────────────────────────────────────────────────────
def make_y9(y: np.ndarray) -> np.ndarray:
    """8종 패턴은 0~7 그대로, 그 밖(패턴 외 세탁 + 정상)은 전부 8.

    회의 결정: 1차 다중분류가 **전 거래**를 받아 9개 중 하나로 보내고, 8로 간 거래만
    2차 이진으로 넘긴다. 따라서 9번째 클래스는 '패턴 외 세탁'만이 아니라
    '8종에 속하지 않는 모든 거래'다 — 정상 거래가 여기 포함된다.

    팀 문서(`EDA 결과와 모델 설계 결정`)는 "패턴 외를 하나의 일관된 9번째 클래스로
    가정하지 않는다"고 경고한 바 있다. 회의 결정이 그 경고를 덮어쓴 것이므로
    경고 자체는 meta 의 `open_risks` 에 그대로 남긴다.
    """
    return np.where((y >= 0) & (y <= 7), y, CLASS8).astype(np.int8)


# ─────────────────────────────────────────────────────────────────────────────
# 실행
# ─────────────────────────────────────────────────────────────────────────────
def run_basis(ns: dict, raw_all: dict, vocab_keys: dict, n_nodes_raw: int,
              basis: str, out_root: Path, clean_info: dict, copy_raw: bool = True) -> dict:
    cfg = BASES[basis]
    out = out_root / f'{ns["DATASET"]}_{basis}'
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    print(f'\n{"=" * 78}\n[{basis}] {cfg["note"]}\n{"=" * 78}')

    # ── 1. 꼬리 절단 · 시간순 분할 (재사용: trim_tails / sort_split_reindex) ──
    ns['TAIL_MIN_FRAC'] = cfg['tail_min_frac']
    # basis 가 하나뿐이면 원본을 그대로 소비한다(1.8억 행 사본 = 9 GB). 둘 이상이면 원래대로 복사.
    if copy_raw:
        raw = {k: v.copy() for k, v in raw_all.items()}
    else:
        raw = dict(raw_all)                             # 배열은 공유, 원래 dict 만 비운다(참조 해제용)
        raw_all.clear()
    raw, trim_info = ns['trim_tails'](raw)
    raw, old2new, split_info = ns['sort_split_reindex'](raw, n_nodes_raw)
    split = raw['split']
    n_rows, n_nodes = len(split), split_info['n_accounts']

    # ── 2. 고립 계좌 마스크: 계산해서 저장하되 **적용하지 않는다** ────────────
    # 2026-08-25 회의의 실험 옵션이다. 이번 회의는 불균형 대응으로 클러스터 기반
    # 언더샘플링(§E)을 택했으므로, 라벨에서 파생된 필터를 겹쳐 적용하면 두 효과가
    # 뒤섞인다. 마스크는 남겨 모델 담당자가 켤 수 있게 한다.
    iso_edge, iso_stats = ns['flag_isolated'](raw, n_nodes)
    iso_edge = iso_edge & (raw['is_pos'] != 1)          # 양성은 어떤 필터로도 빼지 않는다
    sample_mask_isolated = ~(iso_edge & (split == 0))   # train 에만 해당하는 마스크
    del iso_edge

    # ── 3. 패턴 정답지 · 파생 배열 · 라벨 (재사용) ───────────────────────────
    pat = ns['load_patterns'](ns['PATTERNS_PATH'], vocab_keys, old2new,
                              split_info['ts_offset_epoch_min'])
    base_keys = ['ts_min', 'src_id', 'dst_id', 'pair_id', 'paid', 'recv', 'cents',
                 'fmt_code', 'pay_code', 'recv_code', 'split']
    if not LOWMEM:
        base_keys.insert(7, 'recv_cents')
    a = {k: raw[k] for k in base_keys}
    a['is_pos'] = (raw['is_pos'] == 1)
    _day = raw['ts_epoch_min'] // 1440
    a['hour'] = ((raw['ts_epoch_min'] % 1440) // 60).astype(np.int8)
    a['dow'] = ((_day + 3) % 7).astype(np.int8)
    a['ccy_mismatch'] = a['pay_code'] != a['recv_code']
    def _rate(idx):                                     # 노트북 셀 19(실행 셀)의 재현
        names = list(idx)
        miss = [nm for nm in names if nm not in ns['USD_RATE']]
        return np.array([ns['USD_RATE'].get(nm, ns['USD_RATE_FALLBACK'])
                         for nm in names], dtype=np.float64), miss
    pay_rate, pay_miss = _rate(vocab_keys['pay'].index)
    recv_rate, recv_miss = _rate(vocab_keys['recv'].index)
    if LOWMEM:
        # 전체 길이 파생 배열(paid_log·recv_log·usd_paid·usd_recv = 각 1.4 GB)을 만들지 않는다.
        # FeatureBuilderLM 이 블록마다 같은 식으로 즉석 계산한다. amt_mismatch 는 recv_cents 를
        # 버렸으므로 rint(recv*100) 을 청크로 다시 만들어 비교한다(값은 _prep_chunk 와 동일).
        a['_pay_rate'], a['_recv_rate'] = pay_rate, recv_rate
        amt_mm = np.empty(n_rows, dtype=bool)
        for st in range(0, n_rows, 20_000_000):
            en = min(st + 20_000_000, n_rows)
            rc = np.rint(a['recv'][st:en] * 100).astype(np.int64)
            amt_mm[st:en] = (~a['ccy_mismatch'][st:en]) & (a['cents'][st:en] != rc)
        a['amt_mismatch'] = amt_mm
        del amt_mm
    else:
        a['paid_log'] = np.log1p(a['paid'])
        a['recv_log'] = np.log1p(a['recv'])
        a['amt_mismatch'] = (~a['ccy_mismatch']) & (a['cents'] != a['recv_cents'])
        a['usd_paid'] = a['paid'] * pay_rate[a['pay_code']]
        a['usd_recv'] = a['recv'] * recv_rate[a['recv_code']]
    day_idx = (_day - _day.min()).astype(np.int16)
    del _day

    y, attempt, label_stats = ns['attach_labels'](a, pat)
    y9 = make_y9(y)
    is_oop = (y == CLASS8)                              # 패턴 외 '세탁' (정상 아님)
    del raw
    if LOWMEM:
        # cents(int64 1.4 GB)는 라벨 매칭(JOIN_KEYS)까지만 필요하다. 피처는 '만 단위 반올림 금액' bool 만 쓴다.
        ra = np.empty(n_rows, dtype=bool)
        for st in range(0, n_rows, 20_000_000):
            en = min(st + 20_000_000, n_rows)
            ra[st:en] = (a['cents'][st:en] % 10000 == 0)
        a['round_amt'] = ra
        del a['cents'], ra
        # 행 단위 큰 배열(9 GB)을 디스크 memmap 으로 내린다. 피처 계산은 1M 행 연속 블록이라 순차 읽기다.
        # RAM 은 무작위 접근하는 prefix 합·정렬 키가 쓴다(FeatureBuilderLM). 끝나면 파일을 지운다.
        spill_dir = LOWMEM_SCRATCH / f'a_{basis}'
        spill_dir.mkdir(parents=True, exist_ok=True)
        spilled = [k for k, v in a.items() if isinstance(v, np.ndarray) and v.ndim == 1
                   and len(v) == n_rows and k not in ('split', 'is_pos')]
        for k in spilled:
            p = spill_dir / f'{k}.npy'
            np.save(p, a[k])
            a[k] = np.load(p, mmap_mode='r')
        print(f'[lowmem] a 배열 {len(spilled)}개를 디스크 memmap 으로 내림 -> {spill_dir}')
    gc.collect()

    hist9 = {int(c): int((y9 == c).sum()) for c in range(9)}
    print(f'[9class] {hist9} · 8번 클래스 중 세탁 {int(is_oop.sum()):,} / '
          f'정상 {int((y9 == CLASS8).sum() - is_oop.sum()):,}')

    # ── 4. 1차 9-class 학습셋 (전 거래 · 엣지 단위) ───────────────────────────
    ns['a'], ns['vocab_keys'] = a, vocab_keys
    fb = ns['FeatureBuilder'](a, vocab_keys)
    feat_names, feat_blocks = fb.names, fb.blocks
    n_feat = len(feat_names)
    b1, b2 = int(np.searchsorted(split, 1)), int(np.searchsorted(split, 2))
    bounds = [(0, b1), (b1, b2), (b2, n_rows)]

    s1 = out / 'stage1_9class'
    s1.mkdir(exist_ok=True)
    BLK = 1_000_000
    for s, (lo, hi) in enumerate(bounds):
        tag = SPLIT_TAGS[s]
        mm = np.lib.format.open_memmap(s1 / f'X_{tag}.npy', mode='w+',
                                       dtype=np.float32, shape=(hi - lo, n_feat))
        for st in range(lo, hi, BLK):
            en = min(st + BLK, hi)
            mm[st - lo:en - lo] = fb.transform(np.arange(st, en))
        mm.flush(); del mm
        np.save(s1 / f'y9_{tag}.npy', y9[lo:hi])
        np.save(s1 / f'is_pos_{tag}.npy', a['is_pos'][lo:hi].astype(np.int8))
        np.save(s1 / f'is_oop_{tag}.npy', is_oop[lo:hi])
        if tag == 'tr':     # 정의상 train 에서만 False 가 될 수 있다. va/te 는 전부 True 라 안 만든다
            np.save(s1 / 'sample_mask_isolated_tr.npy', sample_mask_isolated[lo:hi])
        h = np.bincount(y9[lo:hi], minlength=9)
        print(f'[stage1] {tag}: {hi - lo:,}x{n_feat} · 패턴 {int(h[:8].sum()):,} '
              f'· 클래스8 {int(h[8]):,} (그중 세탁 {int(is_oop[lo:hi].sum()):,})')

    # 절대 시각 피처 열 위치 — 꼬리 지름길 검증·제거 실험용(§보고서)
    abs_time_cols = [i for i, nm in enumerate(feat_names)
                     if nm in ('hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend')]
    if hasattr(fb, 'close'):                            # lowmem: prefix 합 memmap 파일 정리
        fb.close()
    del fb
    gc.collect()

    # ── 5. 2차 이진 학습셋 (1차가 8로 보낸 거래) ──────────────────────────────
    s2 = out / 'stage2_binary'
    s2.mkdir(exist_ok=True)
    for s, (lo, hi) in enumerate(bounds):
        tag = SPLIT_TAGS[s]
        loc = np.flatnonzero(y9[lo:hi] == CLASS8).astype(np.int64)   # split 내부 인덱스
        np.save(s2 / f'idx_{tag}.npy', loc)
        np.save(s2 / f'y2_{tag}.npy', a['is_pos'][lo:hi][loc].astype(np.int8))
        print(f'[stage2] {tag}: 대상 {len(loc):,} · 양성(패턴외 세탁) '
              f'{int(a["is_pos"][lo:hi][loc].sum()):,}')

    # ── 6. 클러스터 기반 언더샘플링 (train 전용) ──────────────────────────────
    und = out / 'undersample'
    und.mkdir(exist_ok=True)
    y9tr, postr = y9[:b1], a['is_pos'][:b1]
    normal_tr = np.flatnonzero((y9tr == CLASS8) & (~postr))     # 정상만 줄인다
    keep_always = np.flatnonzero((y9tr != CLASS8) | postr)      # 패턴 + 패턴외 세탁
    n_pattern_tr = int((y9tr <= 7).sum())
    oop_tr = np.flatnonzero((y9tr == CLASS8) & postr)
    print(f'[undersample] mode={UNDERSAMPLE_MODE} · train 정상 {len(normal_tr):,}행 · '
          f'패턴 {n_pattern_tr:,}행 · 항상 유지 {len(keep_always):,}행')
    k_sweep, k_best, km_stat, lab, dist = [], None, None, None, None
    urep, s2rep = [], []
    if UNDERSAMPLE_MODE != 'none':
        if UNDERSAMPLE_MODE == 'cluster':
            Xtr = np.load(s1 / 'X_tr.npy', mmap_mode='r')
            Xn = np.asarray(Xtr[normal_tr])
            k_sweep = us.sweep_k(Xn, KMEANS_KS, seed=SEED)
            # K 는 회의 미확정. '양자화 오차가 최적 대비 10% 이내인 가장 작은 K' 규칙으로 고른다
            # — 무조건 큰 K 를 고르면(오차는 K 와 함께 단조 감소) 군집당 표본이 말라 층화가 깨진다.
            _best_err = min(r['inertia_per_row'] for r in k_sweep)
            k_best = int(min(r['k'] for r in k_sweep if r['inertia_per_row'] <= _best_err * 1.10))
            lab, dist, km_stat = us.fit_clusters(Xn, k_best, seed=SEED)
            del Xn, Xtr
            gc.collect()
        rng = np.random.default_rng(SEED)

        def _picks(n_t: int) -> list[tuple[str, np.ndarray]]:
            # 원래 순서(cluster 먼저, random 은 rng 한 번)를 유지해 cluster 모드 산출물이 그대로 재현된다
            out_ = []
            if lab is not None:
                out_.append(('cluster', normal_tr[us.sample_indices(lab, dist, n_t, seed=SEED)]))
            out_.append(('random', normal_tr[np.sort(rng.choice(len(normal_tr), n_t, replace=False))]))
            return out_

        for r in UNDERSAMPLE_RATIOS:
            n_t = min(r * n_pattern_tr, len(normal_tr))
            for nm, pick in _picks(n_t):
                idx = np.sort(np.concatenate([keep_always, pick]))
                np.save(und / f'{nm}_r{r}_tr.npy', idx.astype(np.int64))
                urep.append({'method': nm, 'ratio': r, 'n_normal': len(pick),
                             'n_total': len(idx), 'n_pattern': n_pattern_tr,
                             'n_oop_pos': int(len(oop_tr)),
                             'clusters_covered': int(len(np.unique(lab[np.searchsorted(
                                 normal_tr, pick)]))) if nm == 'cluster' else None})
        pd.DataFrame(urep).to_csv(und / 'variants.csv', index=False)
        if k_sweep:
            pd.DataFrame(k_sweep).to_csv(und / 'kmeans_sweep.csv', index=False)

        # 2차 이진도 불균형이 1차 못지않다(정상 : 패턴 외 세탁). 같은 규칙으로 변형을 만든다.
        for r in UNDERSAMPLE_RATIOS:
            n_t = min(r * len(oop_tr), len(normal_tr))
            for nm, pick in _picks(n_t):
                idx = np.sort(np.concatenate([oop_tr, pick]))
                np.save(und / f'stage2_{nm}_r{r}_tr.npy', idx.astype(np.int64))
                s2rep.append({'stage': 'stage2', 'method': nm, 'ratio': r,
                              'n_normal': len(pick), 'n_pos': len(oop_tr), 'n_total': len(idx)})
        pd.DataFrame(s2rep).to_csv(und / 'variants_stage2.csv', index=False)
        print(f'[undersample] 1차용 {len(urep)}벌 · 2차용 {len(s2rep)}벌 (양성 {len(oop_tr):,}건 전량 유지)')
    else:
        print('[undersample] 생략 (--undersample none)')

    # ── 7. 후처리 블록 — **정답으로 만든 오라클 상한** (§F) ────────────────────
    # 여기 저장되는 블록은 운영 산출물이 아니다. 1차 모델이 완벽하다고 가정했을 때
    # '계좌 공유 + 시간창' 묶기 규칙이 정답 시도를 얼마나 되살리는지 재는 상한선이다.
    # 그래서 파일 이름을 oracle_ 로 붙인다.
    blk = out / 'blocks'
    blk.mkdir(exist_ok=True)
    pos_rows = np.flatnonzero(y9 <= 7)                  # 8종 패턴 거래 = 패턴 블록 후보

    # 채점 자격: (a) 기간 절단으로 잘리지 않은 완결 시도, (b) 거래 2건 이상,
    #            (c) 한 split 안에 온전히 들어온 시도.
    # (b) 를 빼면 거래가 1건만 남은 시도가 어떤 창에서도 자동 성공이 돼 지표가 부풀려진다.
    declared = ns['PATTERN_ATTEMPTS']['n_edges_declared']
    ga = pd.DataFrame({'att': attempt[pos_rows], 'split': split[pos_rows],
                       'cls': y9[pos_rows]}).groupby('att').agg(
        n=('split', 'size'), s_min=('split', 'min'), s_max=('split', 'max'),
        cls=('cls', lambda v: int(v.mode().iloc[0])))
    ga['declared'] = declared.reindex(ga.index).to_numpy()
    ga['complete'] = ga['n'] == ga['declared']
    ga['contained'] = ga['s_min'] == ga['s_max']
    ga['eligible'] = ga['complete'] & ga['contained'] & (ga['n'] >= 2)
    ga.to_csv(blk / 'attempt_eligibility.csv')
    elig_by_split = {s_: ga.index[ga['eligible'] & (ga['s_min'] == s_)].to_numpy()
                     for s_ in range(3)}
    print(f'[block] 시도 {len(ga)} · 완결 {int(ga["complete"].sum())} · '
          f'split 내 완결+2건이상 {int(ga["eligible"].sum())} '
          f'(train {len(elig_by_split[0])} / val {len(elig_by_split[1])} / test {len(elig_by_split[2])})')

    def sweep_one(w: int, s_: int) -> tuple[dict, dict]:
        sel = pos_rows[split[pos_rows] == s_]           # 블록은 split 안에서만 만든다
        comp_ = pb.link_components(a['src_id'][sel], a['dst_id'][sel], a['ts_min'][sel], w)
        ev_ = pb.evaluate_blocks(comp_, attempt[sel], eligible=elig_by_split[s_],
                                 per_attempt=True)
        ev_ = {'n_blocks': 0, 'purity': float('nan'), 'frag_mean': float('nan'),
               'mixed_blocks': 0, 'strict_recovery': float('nan'),
               'attempt_ids': np.zeros(0), 'strict_flags': np.zeros(0, bool), **ev_}
        per_c = {}
        for c in range(8):
            ids = ga.index[(ga['cls'] == c) & ga['eligible'] & (ga['s_min'] == s_)].to_numpy()
            m_ = np.isin(ev_.get('attempt_ids', np.zeros(0)), ids)
            per_c[ns['CLASS_NAMES'][c]] = (float(ev_['strict_flags'][m_].mean())
                                           if m_.any() else float('nan'))
        return ev_, per_c

    sweep = []
    for w in BLOCK_WINDOWS:
        for s_, tag in enumerate(SPLIT_TAGS):
            ev_, per_c = sweep_one(w, s_)
            macro = float(np.nanmean(list(per_c.values()))) if per_c else float('nan')
            sweep.append({'window_min': w, 'split': tag,
                          'n_scored': ev_['n_scored'], 'n_blocks': ev_['n_blocks'],
                          'strict_recovery': round(ev_['strict_recovery'], 4),
                          'macro_strict_8class': round(macro, 4),
                          'purity': round(ev_['purity'], 4),
                          'frag_mean': round(ev_['frag_mean'], 4),
                          'mixed_blocks': ev_['mixed_blocks'],
                          **{k_: round(v_, 4) for k_, v_ in per_c.items()}})
    sw = pd.DataFrame(sweep)
    sw.to_csv(blk / 'window_sweep.csv', index=False)

    # 창은 **train 구간에서만** 고른다. 8종 macro 평균을 최대화하고, 동률이면 좁은 창.
    # (시도 단순평균으로 고르면 다수 클래스가 끌고 가 BIPARTITE 가 0 으로 눌린다 — §보고서)
    tr_sw = sw[sw['split'] == 'tr']
    _top = tr_sw['macro_strict_8class'].max()
    w_best = int(tr_sw.loc[tr_sw['macro_strict_8class'] >= _top - 1e-12, 'window_min'].min())
    print(sw[sw['split'] == 'tr'][['window_min', 'n_scored', 'n_blocks', 'strict_recovery',
                                   'macro_strict_8class', 'purity', 'frag_mean']]
          .to_string(index=False))
    print(f'[block] 채택 창 W={w_best}분 (train 8종 macro 기준)')

    rows_all, comp_all, off = [], [], 0
    for s_ in range(3):
        sel = pos_rows[split[pos_rows] == s_]
        c_ = pb.link_components(a['src_id'][sel], a['dst_id'][sel], a['ts_min'][sel], w_best)
        rows_all.append(sel)
        comp_all.append(c_ + off)
        off += (int(c_.max()) + 1) if len(c_) else 0
    rows_all, comp_all = np.concatenate(rows_all), np.concatenate(comp_all)
    np.savez_compressed(blk / f'oracle_blocks_W{w_best}.npz', rows=rows_all, comp=comp_all,
                        y9=y9[rows_all], attempt=attempt[rows_all], split=split[rows_all],
                        ts=a['ts_min'][rows_all],
                        note=np.array(['정답 라벨로 만든 오라클 블록. 블록은 split 안에서만 '
                                       '구성된다. 운영 산출물이 아니다.']))
    single = np.flatnonzero(is_oop)
    np.save(blk / 'oracle_single_alert_rows.npy', single.astype(np.int64))
    n_blk_total = int(len(np.unique(comp_all)))
    print(f'[block] 오라클 패턴 블록 {n_blk_total:,} · 단건 알림 후보 {len(single):,}')

    # ── 8. BIPARTITE vs STACK 심층 분석 재료 (§G) ─────────────────────────────
    eda = out / 'eda'
    eda.mkdir(exist_ok=True)
    struct = []
    for c in range(8):
        m = np.flatnonzero(y9 == c)
        at = attempt[m]
        ua = np.unique(at[at >= 0])
        comp_inf = pb.link_components(a['src_id'][m], a['dst_id'][m], a['ts_min'][m],
                                      10 ** 9)          # 창 무제한
        ev_inf = pb.evaluate_blocks(comp_inf, at)
        sizes = np.bincount(at[at >= 0] - at[at >= 0].min()) if len(ua) else np.zeros(1)
        sizes = sizes[sizes > 0]
        acc = [len(np.unique(np.concatenate([a['src_id'][m][at == q], a['dst_id'][m][at == q]])))
               for q in ua]
        dur = [int(a['ts_min'][m][at == q].max() - a['ts_min'][m][at == q].min()) for q in ua]
        struct.append({
            'class': c, 'name': ns['CLASS_NAMES'][c], 'n_edges': len(m), 'n_attempts': len(ua),
            'edges_per_attempt_median': float(np.median(sizes)),
            'accounts_per_attempt_median': float(np.median(acc)) if acc else 0.0,
            'duration_min_median': float(np.median(dur)) if dur else 0.0,
            '창무제한_블록수': ev_inf.get('n_blocks', 0),
            '창무제한_시도당조각': round(ev_inf.get('frag_mean', 0.0), 3),
            '창무제한_strict복원율': round(ev_inf.get('strict_recovery', 0.0), 4),
        })
    st_df = pd.DataFrame(struct)
    st_df.to_csv(eda / 'per_class_structure.csv', index=False)
    print('\n[eda] 클래스별 구조 (창 무제한에서도 못 묶이면 후처리로 복원 불가)')
    print(st_df.to_string(index=False))

    # 원시 거래 덤프 — 담당자가 원본 CSV 와 바로 대조할 수 있어야 한다.
    # 재부여된 정수 id·오프셋 시각만 주면 원본을 못 찾으므로 원본 (Bank, Account) 와
    # 원본 timestamp 를 복원해 함께 싣는다. 시도는 앞에서 6개가 아니라 고르게 뽑는다.
    new2old = np.full(n_nodes, -1, dtype=np.int64)
    _old_used = np.flatnonzero(old2new >= 0)
    new2old[old2new[_old_used]] = _old_used
    node_key = vocab_keys['nodes'].index.to_numpy()
    ts0 = split_info['ts_offset_epoch_min']

    def orig_acct(nid: int) -> str:
        o = int(new2old[nid])
        return str(node_key[o]) if o >= 0 else f'<미등재 new_id={nid}>'

    dump = []
    for c in (5, 6):                                    # BIPARTITE, STACK
        m = np.flatnonzero(y9 == c)
        ua_all = np.unique(attempt[m][attempt[m] >= 0])
        pick = ua_all[np.linspace(0, len(ua_all) - 1, min(8, len(ua_all))).astype(int)] \
            if len(ua_all) else ua_all
        for q in pick:
            r = m[attempt[m] == q]
            for i in r[np.argsort(a['ts_min'][r])]:
                dump.append({
                    'class': ns['CLASS_NAMES'][c], 'attempt_id': int(q),
                    'timestamp': str(np.datetime64(0, 'm')
                                     + np.timedelta64(int(a['ts_min'][i]) + ts0, 'm')),
                    'day': int(day_idx[i]),
                    'from_bank_account': orig_acct(int(a['src_id'][i])),
                    'to_bank_account': orig_acct(int(a['dst_id'][i])),
                    'amount_paid': float(a['paid'][i]),
                    'payment_currency': vocab_keys['pay'].index[a['pay_code'][i]],
                    'payment_format': vocab_keys['fmt'].index[a['fmt_code'][i]],
                    'self_loop': bool(a['src_id'][i] == a['dst_id'][i]),
                    'split': SPLIT_TAGS[int(split[i])],
                    'global_row': int(i)})
    dmp = pd.DataFrame(dump).sort_values(['class', 'attempt_id', 'timestamp'])
    dmp.to_csv(eda / 'bipartite_stack_raw.csv', index=False)
    print(f'[eda] 원시 덤프 {len(dmp)}행 / 시도 {dmp["attempt_id"].nunique()}개 '
          f'-> {eda / "bipartite_stack_raw.csv"}')

    # ── 9. 색인 · meta · 자체 검증 ────────────────────────────────────────────
    # lowmem 에서는 ts_min·pair_id 가 int32 다. 파일 스키마는 HI-Small 산출물과 같게 int64 로 저장한다.
    np.savez_compressed(out / 'index_full.npz', ts_min=a['ts_min'].astype(np.int64),
                        src_id=a['src_id'], dst_id=a['dst_id'],
                        pair_id=a['pair_id'].astype(np.int64), split=split,
                        y9=y9, y_raw=y, is_pos=a['is_pos'], attempt=attempt, day=day_idx)

    checks = run_checks(a, y, y9, split, feat_names, s1, s2, und, bounds, basis, label_stats,
                        trim_info, ns['DATASET'])
    meta = {
        'created': time.strftime('%Y-%m-%d %H:%M:%S'), 'dataset': ns['DATASET'],
        'basis': basis, 'basis_note': cfg['note'], 'seed': SEED,
        'meeting': '2026-08-26 — 2단계 계층(1차 9-class 다중분류 / 2차 패턴외 이진)',
        'reused_from': {'notebook': str(nbcells.NB_PATH),
                        'cells': [c for c, _ in nbcells.DEF_CELLS],
                        'cell_sha1': nbcells.fingerprint()},
        'lowmem': ({'enabled': True, 'overrides': list(lm.OVERRIDE_NAMES),
                    'module': 'prep9/lowmem.py',
                    'why': 'cgroup RAM 27.3 GiB 상한. 노트북 원본은 그대로 두고 메모리 배치만 바꾼 함수로 갈아끼움. '
                           'HI-Small 로 기존 산출물과 동치 검증(compare_lowmem_small.py).',
                    'dtype_changes': 'ts_min·pair_id int32 (index_full.npz 저장 시 int64 복원), recv_cents 는 중복 판정 후 폐기'}
                   if LOWMEM else {'enabled': False}),
        'clean': clean_info, 'trim': trim_info, 'split': split_info, 'labels': label_stats,
        'class_hist_9': hist9,
        'class8_composition': {'normal': int(hist9[8] - int(is_oop.sum())),
                               'out_of_pattern_laundering': int(is_oop.sum())},
        'features': {'n': n_feat, 'names': feat_names, 'blocks': feat_blocks,
                     'absolute_time_cols': abs_time_cols},
        'undersample': {'mode': UNDERSAMPLE_MODE,
                        'k_sweep': k_sweep, 'k_chosen': k_best, 'kmeans': km_stat,
                        'ratios': list(UNDERSAMPLE_RATIOS), 'variants': urep,
                        'variants_stage2': s2rep, 'center_frac': 0.5,
                        'scope': ('train split 의 정상 거래만. val/test 는 손대지 않음. '
                                  '1차용(stage1 비율 = 정상:8종패턴)과 2차용(stage2 비율 = '
                                  '정상:패턴외세탁)을 따로 만든다')},
        'blocks': {'window_sweep': sweep, 'window_chosen': w_best,
                   'window_selected_on': 'train split · 8종 macro strict · 동률이면 좁은 창',
                   'n_pattern_blocks': n_blk_total,
                   'n_single_alert_candidates': int(len(single)),
                   'scoring_eligibility': '완결 시도 + 거래 2건 이상 + 한 split 안에 온전히 포함',
                   'attempts_total': int(len(ga)),
                   'attempts_complete': int(ga['complete'].sum()),
                   'attempts_eligible': int(ga['eligible'].sum()),
                   'kind': 'oracle — 정답 라벨로 구성. 운영 산출물이 아니다'},
        'isolated_filter': {
            **iso_stats, 'applied': False,
            'saved_splits': ['tr'],
            'reason': ('마스크만 저장하고 적용하지 않았다. (1) 이번 회의는 불균형 대응으로 '
                       '클러스터 언더샘플링을 택했고 라벨 파생 필터를 겹치면 두 효과가 섞인다. '
                       '(2) 이 마스크는 train/val/test 를 구분하지 않은 전체 그래프에서 '
                       '세탁 라벨로 계산되므로 **미래(val·test) 라벨의 함수**다. 절대 피처로 '
                       '쓰지 말고, 켜면 학습 분포가 서빙 분포와 달라진다는 점을 함께 보고한다.'),
            'derived_from': '전 구간 라벨(미래 정보 포함)'},
        'assumptions': [
            'USD_RATE 는 2026-08-25 노트북의 팀 미확정 가정값을 그대로 물려받았다.',
            (f'클러스터 K={k_best} 는 회의에서 정한 값이 아니라 양자화 오차 스윕으로 골랐다.'
             if k_best is not None else
             f'언더샘플 mode={UNDERSAMPLE_MODE}: 클러스터 층화를 만들지 않았다(08-27 선택이 full 이라 '
             '쓰이는 곳이 없고, 이 세트 크기에서는 클러스터링이 병목이다).'),
            f'블록 시간창 W={w_best}분은 회의 미확정. 후속 검증 9번 대상.',
            '언더샘플 비율 10/30/100/300 은 회의 미확정 — 모델 담당자가 고르라고 4벌 만든다.',
            'center_frac=0.5 (군집 안에서 중심 근접 절반 + 무작위 절반) 는 회의 미확정 값이다. '
            '1.0 으로 올리면 경계 표본이 사라지고 0.0 이면 군집 층화 무작위와 같아진다.',
            'USE_ABSOLUTE_TIME_FEATS=True 를 08-25 노트북에서 그대로 물려받아 hour/dow 5개가 '
            'X 에 들어 있다. 화두 10·17 은 절대 시각을 쓰지 않기로 했으므로 학습 직전에 '
            'features.absolute_time_cols 열을 빼는 것이 팀 결정에 맞는다.',
            f'블록 창 W 는 train 구간의 8종 macro strict 로 골랐다(자격: 완결+2건이상+split 내 포함). '
            f'클래스별 최적 창이 서로 달라 단일 창으로 전부를 만족시킬 수 없다.',
        ],
        'open_risks': [
            '팀 문서 16:05 결정은 "패턴 외를 하나의 일관된 9번째 클래스로 가정하지 않는다"였다. '
            '이번 회의의 9-Class 는 그 경고를 덮어쓴 구조이므로 클래스 8 내부 이질성을 반드시 측정한다.',
            '언더샘플링은 사전확률을 바꾼다. 정밀도는 손대지 않은 val/test 전량에서만 잰다.',
            '2차 이진 학습셋(stage2_binary)은 1차의 예측이 아니라 정답 y9==8 로 모집단을 정한 '
            '**오라클 라우팅**이다. 운영에서는 1차가 8로 오분류한 패턴 거래가 섞여 들어오므로 '
            '분포가 다르다. 종단 성능은 1차 출력을 실제로 연결해 따로 재야 한다.',
            'blocks/ 의 블록과 단건 후보도 정답으로 만든 오라클 상한이다. 파일명 oracle_ 접두사 '
            '그대로 읽고, 운영 알림 성능으로 인용하지 않는다.',
            'evaluate_blocks 의 순도는 라벨된 시도끼리의 섞임만 잰다. 1차 오탐(정상 거래)이 '
            '블록에 섞이는 몫은 여기 안 잡히므로 실제 알림 순도는 이보다 낮다.',
        ],
        'checks': checks, 'elapsed_s': round(time.time() - t0, 1),
    }
    (out / 'features_meta.json').write_text(
        json.dumps(meta, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
    if LOWMEM:
        for p in (LOWMEM_SCRATCH / f'a_{basis}').glob('*.npy'):
            try:
                p.unlink()
            except OSError:
                pass
    print(f'[done] {basis} — {time.time() - t0:,.1f}s -> {out}')
    return meta


def run_checks(a, y, y9, split, feat_names, s1, s2, und, bounds, basis, label_stats,
               trim_info, dataset: str = 'HI-Small') -> list[dict]:
    """전처리가 팀이 이미 합의한 사실을 재현하는지, 누수가 없는지 자체 점검.

    18day 전용이던 팀 EDA 대조를 두 basis 모두에서 돌게 고쳤다(2026-08-27).
    10day는 꼬리 8일을 잘라내므로 팀 EDA 원값(5,177/3,209/1,968)과 원시 카운트가
    그대로 맞지 않는다 — 다만 '패턴 외' 1,968만은 전량이 1~10일 안에 있어 트림과
    무관하게 두 basis 모두 그대로 성립한다(재사용 노트북 셀 35의 보정식과 동일한 사실).
    거기에 unmatched_keys == positives_cut 항등식을 추가한다 — 이건 트림으로 잘려나간
    양성 행 수와, 그래서 패턴 매칭이 안 된 정답지 키 수가 같아야 한다는 뜻이라 계좌
    재부여(old2new)나 시간 오프셋이 어긋나면 두 basis 어느 쪽에서도 깨진다.
    """
    out = []

    def chk(name, ok, got, want=''):
        out.append({'검사': name, '통과': bool(ok), '실측': got, '기대': want})

    # 트림 정합은 세트와 무관한 항등식이라 항상 검사한다.
    chk('트림 정합 — unmatched_keys == positives_cut',
        label_stats['unmatched_keys'] == trim_info['positives_cut'],
        f"{label_stats['unmatched_keys']} vs {trim_info['positives_cut']}",
        '같아야 함 (꼬리로 잘린 양성 수 = 패턴 매칭 실패 키 수)')

    # 팀 EDA 대조는 HI-Small 실측 상수라 다른 세트에는 적용할 수 없다.
    # 상수를 그대로 들고 가면 정상인 실행이 FAIL 로 뜬다 — 건너뛰되 건너뛴 사실을 남긴다.
    exp = EDA_EXPECTED.get(dataset)
    if exp is None:
        chk(f'팀 EDA 대조 — {dataset} 기준값 없음(건너뜀)', True, '미대조',
            'EDA_EXPECTED 에 그 세트의 실측값을 넣으면 대조한다')
    else:
        is_full = BASES[basis]['tail_min_frac'] == 0.0        # 꼬리를 자르지 않은 기준
        chk('팀 EDA — 패턴 외 (트림 무관, 전량 주 기간 내)',
            label_stats['positives_out_of_pattern'] == exp['out_of_pattern'],
            label_stats['positives_out_of_pattern'], exp['out_of_pattern'])
        if is_full:
            chk('팀 EDA — 라벨1 총계', int(a['is_pos'].sum()) == exp['label1_total'],
                int(a['is_pos'].sum()), exp['label1_total'])
            chk('팀 EDA — 8종 소속', label_stats['positives_in_pattern'] == exp['in_pattern'],
                label_stats['positives_in_pattern'], exp['in_pattern'])
        else:
            chk('팀 EDA — 8종 소속 (트림 보정: in_pattern + unmatched_keys)',
                label_stats['positives_in_pattern'] + label_stats['unmatched_keys'] == exp['in_pattern'],
                label_stats['positives_in_pattern'] + label_stats['unmatched_keys'], exp['in_pattern'])
            chk('팀 EDA — 라벨1 총계 (트림 보정: positives + positives_cut)',
                int(a['is_pos'].sum()) + trim_info['positives_cut'] == exp['label1_total'],
                int(a['is_pos'].sum()) + trim_info['positives_cut'], exp['label1_total'])
    chk('9-class 정의: 8종은 0~7 그대로',
        bool(np.array_equal(y9[y >= 0], np.minimum(y[y >= 0], 8))), '일치', 'y in 0..7 -> y9 == y')
    chk('9-class 정의: 그 밖은 전부 8',
        bool((y9[(y < 0) | (y == 8)] == 8).all()), '일치', '정상·패턴외 -> 8')
    chk('2차 대상 = 클래스8 전량', int((y9 == 8).sum()) == int(((y < 0) | (y == 8)).sum()),
        int((y9 == 8).sum()), int(((y < 0) | (y == 8)).sum()))
    chk('라벨 파생 컬럼이 피처에 없음',
        not any(k in ' '.join(feat_names) for k in
                ('label', 'laundering', 'pattern', 'attempt', 'y9', 'is_pos')),
        '없음', '피처명에 라벨·패턴·시도 관련 문자열 없음')

    # 분할 경계는 '엄격히' 벌어져야 한다. 같은 분이 두 split 에 걸치면 동시 거래가 갈린다.
    t0m, t1m = int(a['ts_min'][split == 0].max()), int(a['ts_min'][split == 1].min())
    t1M, t2m = int(a['ts_min'][split == 1].max()), int(a['ts_min'][split == 2].min())
    chk('시간순 분할 — 경계가 엄격히 벌어짐(같은 분이 두 split 에 없음)',
        (t0m < t1m) and (t1M < t2m), f'tr_max {t0m} < va_min {t1m} · va_max {t1M} < te_min {t2m}',
        'train.max < val.min 이고 val.max < test.min')

    # as-of 인과성: 세 split 모두에서 확인한다(train 앞부분만 보면 뒷구간을 못 잡는다)
    ci = {nm: i for i, nm in enumerate(feat_names)}
    for k_, tag in enumerate(SPLIT_TAGS):
        X = np.load(s1 / f'X_{tag}.npy', mmap_mode='r')
        take = np.linspace(0, X.shape[0] - 1, min(200_000, X.shape[0])).astype(np.int64)
        Xs = np.asarray(X[take])
        first = Xs[:, ci['dt_src_first']] == 1
        mx = float(Xs[first, ci['src_out_cnt_24h']].max()) if first.any() else 0.0
        chk(f'엄격 과거(as-of) — 첫 송금에 과거 건수 0 [{tag}]', mx == 0.0, mx, 0.0)
        chk(f'피처 NaN/inf 없음 [{tag}]', bool(np.isfinite(Xs).all()), '유한', '유한')
        del X, Xs

    # 인덱스 좌표계: 저장한 인덱스가 자기 split 범위 안에 있고 올바른 행을 가리키는가
    for k_, tag in enumerate(SPLIT_TAGS):
        lo, hi = bounds[k_]
        loc = np.load(s2 / f'idx_{tag}.npy')
        y2 = np.load(s2 / f'y2_{tag}.npy')
        okr = (len(loc) == 0) or (loc.min() >= 0 and loc.max() < hi - lo)
        chk(f'2차 인덱스가 split 내부 좌표 [{tag}]', okr and (y9[lo:hi][loc] == 8).all(),
            f'0~{int(loc.max()) if len(loc) else -1} / 상한 {hi - lo - 1}', 'split 범위 내 · y9==8')
        chk(f'2차 라벨 = 원본 Is Laundering [{tag}]',
            bool(np.array_equal(y2, a['is_pos'][lo:hi][loc].astype(np.int8))), '일치', '일치')
    n_tr = bounds[0][1] - bounds[0][0]
    ytr, ptr = y9[:n_tr], a['is_pos'][:n_tr]
    must = np.flatnonzero((ytr <= 7) | ptr)
    for f_ in sorted(und.glob('*_tr.npy')):
        i_ = np.load(f_)
        stage2 = f_.name.startswith('stage2_')
        need = np.flatnonzero((ytr == 8) & ptr) if stage2 else must
        chk(f'언더샘플 인덱스 [{f_.name}]',
            bool(i_.min() >= 0 and i_.max() < n_tr and np.isin(need, i_).all()),
            f'0~{int(i_.max())} / 상한 {n_tr - 1} · 필수 {len(need):,}건 포함',
            'train 내부 좌표 · 양성 전량 유지')
    for r in out:
        print(f'  [{"OK " if r["통과"] else "FAIL"}] {r["검사"]}: {r["실측"]} (기대 {r["기대"]})')
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--dataset', default='HI-Small',
                    help='HI-Small / HI-Medium / HI-Large 등. 세트는 합치지 않는다(화두 7)')
    ap.add_argument('--basis', nargs='+', default=None, choices=list(BASES),
                    help='기본: HI-Small 은 10day·18day, 그 밖은 main·full')
    ap.add_argument('--scale', default='auto', choices=('auto', 'in_memory', 'chunked'),
                    help="auto 면 파일 크기로 고른다(2GiB 초과 -> chunked)")
    ap.add_argument('--out', default='/workspace/processed_9class')
    ap.add_argument('--undersample', default='cluster', choices=('cluster', 'random', 'none'),
                    help="cluster=원래 동작 / random=무작위 4벌만(피처 행렬 미적재) / none=생략")
    ap.add_argument('--max-chunks', type=int, default=0,
                    help='스모크용: chunked 적재를 앞 N 청크(N×2,000,000행)에서 멈춘다. '
                         '0 이면 전량. 켜면 --out 을 기본 경로와 다르게 줘야 한다.')
    ap.add_argument('--lowmem', action='store_true',
                    help='메모리 절약판(prep9/lowmem.py)으로 적재·정제·정렬·FeatureBuilder 를 갈아끼운다. '
                         'cgroup 27 GiB 상한용. 값은 동일(HI-Small 동치 검증).')
    args = ap.parse_args()
    global UNDERSAMPLE_MODE, LOWMEM, LOWMEM_SCRATCH
    UNDERSAMPLE_MODE = args.undersample
    LOWMEM = args.lowmem
    LOWMEM_SCRATCH = Path(args.out) / '_lowmem_scratch'
    if args.max_chunks and Path(args.out).resolve() == Path('/workspace/processed_9class').resolve():
        ap.error('--max-chunks 는 스모크 전용이다. 정식 산출물 경로를 덮어쓰지 않도록 --out 을 따로 준다')

    basis_list = args.basis or DEFAULT_BASIS.get(args.dataset, DEFAULT_BASIS_OTHER)
    for b in basis_list:                     # 기간을 이름에 박은 별칭은 그 세트에서만 허용
        only = BASES[b].get('only')
        if only and only != args.dataset:
            ap.error(f"basis '{b}' 는 {only} 전용 이름이다. {args.dataset} 에는 "
                     f"'main'/'full' 을 쓴다 — 기간이 다른데 이름만 같으면 산출물이 거짓말을 한다")

    print(f'[reuse] 2026-08-25 검증 통과 전처리 코드 적재 · dataset={args.dataset}')
    # 노트북을 수정하지 않고 네임스페이스에서 갈아끼운다(sha1 게이트 유지). 셀 4 가
    # 이 값으로 TRANS_PATH·PATTERNS_PATH 와 SCALE_MODE 를 확정한다.
    ns = nbcells.load(overrides={'DATASET': args.dataset})
    # 안전장치: 재사용 셀의 OUT_DIR 은 공용 산출물 /workspace/processed_multiclass 를 가리킨다.
    # in_memory 경로에서는 아무것도 쓰지 않지만, 실수로도 덮어쓰지 못하게 즉시 우회한다.
    ns['OUT_DIR'] = Path(args.out) / '_nb_scratch'
    ns['OUT_DIR'].mkdir(parents=True, exist_ok=True)
    if LOWMEM:
        scratch = Path(args.out) / '_lowmem_scratch'
        ns.update(lm.make_overrides(ns, scratch))
        ns['_LOWMEM_MAX_CHUNKS'] = args.max_chunks
        ns['SCALE_MODE'] = 'chunked'                    # lowmem 적재는 parquet 파트 경로만 있다
        print(f'[lowmem] override: {lm.OVERRIDE_NAMES} · scratch={scratch}')
    if args.max_chunks and not LOWMEM:
        # 노트북 셀은 못 고치므로(sha1 게이트) 그 셀이 보는 `pd` 만 감싼다: chunksize 로 읽을 때
        # 앞 N 청크에서 멈춘다. 나머지 pandas 기능은 그대로 통과한다.
        import itertools

        class _PdShim:
            def __init__(self, real, n):
                self._pd, self._n = real, n

            def __getattr__(self, k):
                return getattr(self._pd, k)

            def read_csv(self, *a, **k):
                it = self._pd.read_csv(*a, **k)
                return itertools.islice(it, self._n) if k.get('chunksize') else it

        ns['pd'] = _PdShim(pd, args.max_chunks)
        print(f'[smoke] chunked 적재를 앞 {args.max_chunks} 청크에서 멈춘다 -> {args.out}')
    t0 = time.time()
    # 적재 모드를 'in_memory' 로 못박으면 HI-Large(1.8억 행)에서 죽는다. 셀 4 가 파일 크기로
    # 정해 둔 SCALE_MODE 를 그대로 따르고, --scale 로만 덮어쓴다.
    mode = ns['SCALE_MODE'] if args.scale == 'auto' else args.scale
    print(f'[load] mode={mode} · {ns["TRANS_PATH"]}')
    raw_all, vocab_keys = ns['load_compact'](ns['TRANS_PATH'], mode)
    raw_all, clean_info = ns['clean_rows'](raw_all)
    n_nodes_raw = len(vocab_keys['nodes'])
    print(f'[load] {time.time() - t0:,.1f}s')

    out_root = Path(args.out)
    out_root.mkdir(parents=True, exist_ok=True)
    metas = {}
    for b in basis_list:
        metas[b] = run_basis(ns, raw_all, vocab_keys, n_nodes_raw, b, out_root, clean_info,
                             copy_raw=(len(basis_list) > 1))
    (out_root / 'run_summary.json').write_text(
        json.dumps({b: {k: m[k] for k in ('trim', 'split', 'class_hist_9',
                                          'class8_composition', 'blocks', 'checks')}
                    for b, m in metas.items()}, ensure_ascii=False, indent=2, default=str),
        encoding='utf-8')
    print(f'\n[all done] {time.time() - t0:,.1f}s -> {out_root}')


if __name__ == '__main__':
    main()


## 2. 검증된 전처리 재사용 로더 — `prep9/nbcells.py`

`preprocess_9class.py`가 새로 짜지 않고 그대로 물려받는 부분(로딩·정제·트림·피처 정의 등)을
2026-08-25 검증 통과 노트북에서 셀 단위로 가져오는 모듈. 셀 내용이 바뀌면 sha1 불일치로 즉시 멈춘다
(조용히 다른 코드를 실행하는 사고 방지).

원본 경로: `prep9/nbcells.py`

In [ ]:
"""검증된 전처리 코드 재사용 로더.

`preprocess_multiclass.ipynb`(2026-08-25 산출, 팀 EDA 수치 5개 재현 검증 통과)의
**정의 셀만** 골라 하나의 네임스페이스로 exec 한다. 실행 셀은 재사용하지 않고
`preprocess_9class.py` 가 직접 순서를 짠다 — 9-class 라벨 구성이 달라졌기 때문이다.

셀 번호가 밀리면 조용히 다른 코드를 실행하게 되므로, 셀 소스의 sha1 을 박아두고
불일치하면 즉시 멈춘다.
"""
from __future__ import annotations

import hashlib
import json
from pathlib import Path

NB_PATH = Path('/workspace/notebooks/01_현행_파이프라인/preprocess_multiclass.ipynb')

# (셀 번호, 그 셀이 정의하는 것) — 정의 전용 셀만 싣는다.
DEF_CELLS = [
    (2,  'import·설정 상수'),
    (4,  'extract_member / check_csv_complete + 원본 경로 확정'),
    (6,  '상수(TAIL_MIN_FRAC·CLASS_MAP·USD_RATE·JOIN_KEYS…)'),
    (8,  'KeyDict / _prep_chunk / load_compact / clean_rows'),
    (11, 'trim_tails / sort_split_reindex'),
    (14, 'FILTERS / flag_isolated / flag_special_hub / flag_extreme_amount'),
    (17, 'load_patterns / attach_labels'),
    (21, 'PastIndex'),
    (23, 'fit_code_cols / fit_group_stats / group_z / one_hot'),
    (24, 'FeatureBuilder'),
    (28, 'build_events_attempt / build_events_window / label_events'),
    (29, '_longest_path / _is_bipartite / event_features'),
]

# 2026-08-26 기준 지문. 노트북이 바뀌면 여기서 멈추고 사람이 확인한다.
EXPECTED_SHA1 = {
    2: '12490948',     4: '3db91d9c',     6: '9bd8c401',     8: '22e28e9b',     11: '78b43593',
    14: '1539462e',     17: '3934e1e8',     21: '4e0a923f',     23: '65b20dba',     24: '26f7fd95',
    28: 'd38ea47b',     29: '881de3f7', 
}


def cell_sources(nb_path: Path = NB_PATH) -> dict[int, str]:
    nb = json.loads(Path(nb_path).read_text(encoding='utf-8'))
    return {i: ''.join(c['source']) for i, c in enumerate(nb['cells'])}


def fingerprint(nb_path: Path = NB_PATH) -> dict[int, str]:
    src = cell_sources(nb_path)
    return {i: hashlib.sha1(src[i].encode()).hexdigest()[:8] for i, _ in DEF_CELLS}


def load(strict: bool = True, verbose: bool = True,
         overrides: dict | None = None) -> dict:
    """정의 셀을 exec 한 네임스페이스를 돌려준다.

    `overrides` 는 **매 셀 exec 직후** 다시 덮어쓴다. 노트북 셀 2가 `DATASET='HI-Small'` 을
    정의하고 셀 4가 그 값으로 `TRANS_PATH`·`PATTERNS_PATH` 를 확정하므로, 셀 사이에서
    갈아끼워야 다른 데이터셋의 경로가 잡힌다. 노트북 원본은 건드리지 않으니
    sha1 게이트도 그대로 통과한다 — 데이터셋을 바꾸려고 노트북을 수정하면 안 된다.
    """
    src = cell_sources()
    fp = fingerprint()
    if strict:
        bad = {i: (EXPECTED_SHA1.get(i), fp[i]) for i, _ in DEF_CELLS
               if EXPECTED_SHA1.get(i) != fp[i]}
        if bad:
            raise RuntimeError(
                '재사용 대상 노트북 셀이 바뀌었다. 셀 번호가 밀렸거나 내용이 수정됐다.\n'
                f'  불일치: {bad}\n'
                '  사람이 셀 내용을 확인한 뒤 nbcells.EXPECTED_SHA1 을 갱신한다.')
    ns: dict = {'__name__': '__main__', 'display': lambda *a, **k: None,
                'get_ipython': lambda: None}
    ov = dict(overrides or {})
    for i, what in DEF_CELLS:
        code = src[i]
        if code.lstrip().startswith('!'):
            continue
        exec(compile(code, f'<preprocess_multiclass.ipynb cell {i}>', 'exec'), ns)
        ns.update(ov)                      # 셀이 되돌려 놓은 값을 매번 다시 덮어쓴다
        if verbose:
            print(f'  [nb cell {i:>2}] {what}')
    if ov and verbose:
        print(f'  [override] {ov}')
    return ns


## 3. 클러스터 기반 언더샘플링 — `prep9/undersample.py`

08-26 회의 결정 3("정상 표본을 무작정 삭제하지 않고 대표값 위주로 정제")의 실제 구현.
MiniBatchKMeans로 정상 거래를 K개 군집으로 나눈 뒤 군집 크기 비례 층화 추출,
군집 안에서는 중심 근접 `center_frac` 만큼 + 나머지 무작위.

원본 경로: `prep9/undersample.py`

In [ ]:
"""클러스터 기반 언더샘플링 — 2026-08-26 회의 결정 3.

회의 문구: "정상 표본을 무작정 삭제하지 않고 대표값 위주로 정제하여 학습 안정성을 확보".

구현: 표준화한 피처 공간에서 MiniBatchKMeans 로 정상 거래를 K개 군집으로 나눈 뒤,
군집 크기에 비례해 각 군집에서 뽑는다(층화 추출). 군집 안에서는 중심에 가까운 순으로
`center_frac` 만큼을 대표값으로 먼저 채우고 나머지는 무작위로 채운다.

지켜야 할 것 세 가지 — 어기면 수치가 무의미해진다.
  1. **train split 에만 적용한다.** val/test 의 행을 지우면 평가 분포가 바뀌어
     다른 실험과 비교가 성립하지 않는다.
  2. **표준화 통계와 군집 중심은 train 으로만 적합한다.**
  3. **같은 크기의 무작위 대조군을 함께 만든다.** 대조군이 없으면
     "클러스터가 낫다"는 주장을 검증할 수 없다.

언더샘플링은 사전확률(prior)을 바꾼다. 언더샘플한 셋으로 잰 정밀도는 운영 정밀도가
아니므로, 평가는 반드시 손대지 않은 val/test 전량에서 한다.
"""
from __future__ import annotations

import numpy as np

__all__ = ['fit_clusters', 'sample_indices', 'sweep_k']


def _standardize(X: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    return ((X - mean) / (std + 1e-6)).astype(np.float32, copy=False)


def fit_clusters(X: np.ndarray, k: int, seed: int = 42, batch: int = 20_000,
                 fit_subsample: int | None = 1_000_000):
    """train 정상 행 -> (군집 배정, 중심까지 거리, 적합 통계).

    중심은 `fit_subsample` 행으로 적합하고 배정은 전 행에 한다. 중심 위치는 표본
    100만 행이면 충분히 안정적이고, 전 행 적합은 시간만 몇 배로 든다.
    """
    from sklearn.cluster import MiniBatchKMeans
    mean, std = X.mean(0), X.std(0)
    Z = _standardize(X, mean, std)
    km = MiniBatchKMeans(n_clusters=k, random_state=seed, batch_size=batch,
                         n_init=3, max_iter=100, reassignment_ratio=0.01)
    if fit_subsample is not None and len(Z) > fit_subsample:
        sub = np.random.default_rng(seed).choice(len(Z), fit_subsample, replace=False)
        km.fit(Z[sub])
        lab = km.predict(Z)
    else:
        lab = km.fit_predict(Z)
    d = np.linalg.norm(Z - km.cluster_centers_[lab], axis=1)
    n_fit = int(min(len(X), fit_subsample or len(X)))
    return lab.astype(np.int32), d.astype(np.float32), {
        'k': int(k), 'seed': int(seed), 'inertia': float(km.inertia_),
        'inertia_per_row': float(km.inertia_ / max(n_fit, 1)),   # 적합에 쓴 행 수로 나눈다
        'n_rows_assigned': int(len(X)),
        'n_rows_fit': int(min(len(X), fit_subsample or len(X))),
        'empty_clusters': int(k - len(np.unique(lab))),
        'cluster_size_min': int(np.bincount(lab, minlength=k).min()),
        'cluster_size_max': int(np.bincount(lab, minlength=k).max()),
    }


def sample_indices(lab: np.ndarray, dist: np.ndarray, n_target: int,
                   center_frac: float = 0.5, seed: int = 42) -> np.ndarray:
    """군집 크기 비례 층화 추출. 군집 내부는 (중심 근접 대표값 + 무작위) 혼합."""
    rng = np.random.default_rng(seed)
    n = len(lab)
    if n_target >= n:
        return np.arange(n)
    k = int(lab.max()) + 1
    sizes = np.bincount(lab, minlength=k)
    quota = np.floor(sizes * (n_target / n)).astype(np.int64)
    short = n_target - int(quota.sum())
    if short > 0:                                   # 남은 몫은 큰 군집부터 1개씩
        for c in np.argsort(sizes)[::-1][:short]:
            quota[c] += 1
    order = np.argsort(lab, kind='stable')
    offs = np.searchsorted(lab[order], np.arange(k + 1))
    out = []
    for c in range(k):
        mem = order[offs[c]:offs[c + 1]]
        q = int(min(quota[c], len(mem)))
        if q == 0:
            continue
        n_center = int(round(q * center_frac))
        near = mem[np.argsort(dist[mem], kind='stable')[:n_center]]
        rest = np.setdiff1d(mem, near, assume_unique=False)
        extra = rng.choice(rest, size=min(q - n_center, len(rest)), replace=False) \
            if q > n_center and len(rest) else np.zeros(0, dtype=mem.dtype)
        out.append(np.concatenate([near, extra]))
    return np.sort(np.concatenate(out)) if out else np.zeros(0, dtype=np.int64)


def sweep_k(X: np.ndarray, ks, seed: int = 42, subsample: int | None = 400_000) -> list[dict]:
    """K 후보별 양자화 오차. K 를 감으로 고르지 않기 위한 표."""
    rng = np.random.default_rng(seed)
    Xs = X if subsample is None or len(X) <= subsample else \
        X[rng.choice(len(X), subsample, replace=False)]
    rows = []
    for k in ks:
        _, _, st = fit_clusters(Xs, k, seed=seed)
        rows.append(st)
        print(f'  [kmeans] K={k:>5} inertia/row={st["inertia_per_row"]:.4f} '
              f'빈군집={st["empty_clusters"]} 크기 {st["cluster_size_min"]}~{st["cluster_size_max"]}')
    return rows


## 4. 블록(사건) 후처리 — `prep9/postprocess_blocks.py`

거래별 예측 점수를 "계좌 공유 + 시간창 이내"의 이행적 연결 컴포넌트로 묶어 알림 블록을 만드는 모듈.
라벨을 쓰지 않으므로 운영 추론 시점에 그대로 쓸 수 있고, 정답 라벨을 넣으면 평가용 기준 블록(`evaluate_blocks`)이 나온다.
`preprocess_9class.py`의 §7(오라클 블록)이 이 모듈을 그대로 쓴다.

원본 경로: `prep9/postprocess_blocks.py`

In [ ]:
"""후처리 프로그램 — 거래별 예측을 '알림 블록'으로 묶는다.

2026-08-26 회의 결정 2·3:
  - 1차 9-class 에서 8종 패턴으로 적발된 거래 -> **패턴 블록 단위 알림**
  - 2차 이진에서 적발된 패턴 외 거래        -> **단건 블록 알림**
  - 트리 모델·GNN 모두 거래별 점수만 낸다. 연관 거래를 묶는 것은 별도 후처리의 몫이다.

이 모듈은 라벨을 쓰지 않는다. 입력은 '어떤 거래가 무엇으로 판정됐는가' 뿐이라
운영 추론 시점에 그대로 돌릴 수 있고, 정답 라벨을 넣으면 평가용 기준 블록이 나온다.

묶는 규칙: **계좌 공유 + window 분 이내**의 이행적 연결 컴포넌트.
  A-B 가 창 안이고 B-C 가 창 안이면 A 와 C 가 멀어도 한 블록이다.
"""
from __future__ import annotations

import numpy as np
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

__all__ = ['link_components', 'make_alerts', 'block_frame', 'evaluate_blocks']


def link_components(src: np.ndarray, dst: np.ndarray, ts: np.ndarray,
                    window_min: int, exclude: np.ndarray | None = None) -> np.ndarray:
    """거래들을 '계좌 공유 + window 분 이내'로 이어붙인 연결 컴포넌트 ID.

    엣지 하나를 (src, dst) 두 개의 (계좌, 시각) 사건으로 펼친 뒤 같은 계좌 안에서
    시각순 인접쌍만 잇는다. 완전 그래프를 만들지 않으므로 O(n log n) 이다.

    exclude : 계좌 id 로 색인되는 bool 배열. True 인 계좌는 **다리로 쓰지 않는다**
              (화두 13 특수 허브 — HI-Small 에서는 은행 070 의 15개 계좌가 거래 8.9% 에
              닿는다). 그 계좌에 닿은 거래는 반대쪽 계좌로만 이어지며, 양끝이 모두 제외
              계좌면 단독 블록이 된다. 2026-09-02 추가, 기본값 None 이면 원래 동작.
    """
    m = len(src)
    if m == 0:
        return np.zeros(0, dtype=np.int32)
    ent = np.concatenate([src, dst])
    eid = np.tile(np.arange(m, dtype=np.int32), 2)
    t = np.concatenate([ts, ts])
    if exclude is not None:
        keep = ~np.asarray(exclude, dtype=bool)[ent]
        ent, eid, t = ent[keep], eid[keep], t[keep]
        if len(ent) == 0:
            return np.arange(m, dtype=np.int32)
    o = np.lexsort((t, ent))
    ent_s, eid_s, t_s = ent[o], eid[o], t[o]
    link = (ent_s[1:] == ent_s[:-1]) & ((t_s[1:] - t_s[:-1]) <= window_min)
    rows, cols = eid_s[:-1][link], eid_s[1:][link]
    g = coo_matrix((np.ones(len(rows), dtype=np.int8), (rows, cols)), shape=(m, m))
    _, comp = connected_components(g, directed=False)
    return comp.astype(np.int32)


def make_alerts(pred_class: np.ndarray, pred_binary: np.ndarray | None,
                src: np.ndarray, dst: np.ndarray, ts: np.ndarray,
                window_min: int, row_id: np.ndarray | None = None) -> dict:
    """거래별 판정 -> 알림 목록.

    pred_class : 각 거래의 1차 9-class 예측(0~7 = 8종 패턴, 8 = 패턴 외)
    pred_binary: 패턴 외로 간 거래의 2차 이진 예측(1 = 세탁). None 이면 단건 알림 없음.
                 pred_class 와 같은 길이이며 8 이 아닌 행의 값은 무시한다.
    반환: {'pattern': {...}, 'single': {...}} — 두 알림 종류를 분리해 돌려준다.
    """
    n = len(pred_class)
    row_id = np.arange(n, dtype=np.int64) if row_id is None else np.asarray(row_id)
    hit = np.flatnonzero((pred_class >= 0) & (pred_class <= 7))

    comp = link_components(src[hit], dst[hit], ts[hit], window_min)
    n_blk = int(comp.max()) + 1 if len(comp) else 0
    order = np.argsort(comp, kind='stable')
    offs = np.searchsorted(comp[order], np.arange(n_blk + 1))
    pattern = {
        'member_rows': row_id[hit][order],          # 블록 순서로 정렬된 구성 거래
        'offsets': offs,                            # 블록 b = member_rows[offs[b]:offs[b+1]]
        'n_blocks': n_blk,
        'block_class': _modal_class(pred_class[hit][order], offs, n_blk),
        'window_min': window_min,
    }

    if pred_binary is None:
        single_rows = np.zeros(0, dtype=np.int64)
    else:
        sm = (pred_class == 8) & (np.asarray(pred_binary) == 1)
        single_rows = row_id[np.flatnonzero(sm)]
    single = {'member_rows': single_rows, 'n_blocks': len(single_rows),
              'unit': '단건(거래 1건 = 알림 1건)'}
    return {'pattern': pattern, 'single': single}


def _modal_class(cls_sorted: np.ndarray, offs: np.ndarray, n_blk: int) -> np.ndarray:
    """블록 라벨 = 구성 거래 예측 클래스의 최빈값."""
    if n_blk == 0:
        return np.zeros(0, dtype=np.int8)
    blk = np.repeat(np.arange(n_blk), np.diff(offs))
    cnt = np.bincount(blk.astype(np.int64) * 8 + cls_sorted.astype(np.int64),
                      minlength=n_blk * 8).reshape(n_blk, 8)
    return cnt.argmax(axis=1).astype(np.int8)


def block_frame(alerts: dict, ts: np.ndarray, src: np.ndarray,
                dst: np.ndarray) -> 'pd.DataFrame':
    """패턴 블록 요약 표(알림 화면에 그대로 올릴 최소 필드).

    `ts`/`src`/`dst` 는 `make_alerts` 에 넘긴 `row_id` 로 색인되는 배열이어야 한다.
    (`row_id` 를 생략했다면 make_alerts 에 넘긴 것과 같은 순서의 전체 배열)
    """
    import pandas as pd
    p = alerts['pattern']
    mem, offs = p['member_rows'], p['offsets']
    rows = []
    for b in range(p['n_blocks']):
        s = mem[offs[b]:offs[b + 1]]                 # 구성 거래의 row_id
        rows.append({'block_id': b, 'block_class': int(p['block_class'][b]),
                     'n_edges': len(s),
                     'n_accounts': int(len(np.unique(np.concatenate([src[s], dst[s]])))),
                     'ts_start': int(ts[s].min()), 'ts_end': int(ts[s].max()),
                     'span_min': int(ts[s].max() - ts[s].min())})
    return pd.DataFrame(rows)


def evaluate_blocks(comp: np.ndarray, attempt: np.ndarray,
                    eligible: np.ndarray | None = None,
                    per_attempt: bool = False) -> dict:
    """묶기 규칙이 정답 '시도'를 얼마나 되살리는지 잰다(정답이 있을 때만 쓴다).

    strict_recovery : 그 시도의 (여기 들어온) 거래 전부가 정확히 한 블록에 있고, 그 블록에
                      다른 시도의 거래가 섞이지 않은 시도의 비율.
    purity          : 블록 안에서 최빈 시도가 차지하는 비율의 가중 평균.
    fragmentation   : 시도 하나가 쪼개진 블록 수.

    두 가지를 반드시 알고 써야 한다.

    1. `attempt < 0` 인 행은 **계산 전에 버려진다.** 이 함수는 '라벨된 시도끼리 어떻게
       묶이는가'만 재며, 블록에 섞인 오탐(정상 거래)은 순도에 반영되지 않는다.
       따라서 여기 나오는 순도는 **운영 알림의 순도가 아니라 묶기 규칙의 상한**이다.
    2. 거래가 1건만 남은 시도는 어떤 창에서도 자동으로 strict 성공이 된다(블록도 1개다).
       기간 절단으로 잘린 시도가 섞이면 지표가 부풀려지므로, 채점 대상을 `eligible`
       (완결 + 2건 이상)로 제한해서 부른다.

    eligible : 채점에 넣을 attempt id 배열. None 이면 전부. 블록 구성 자체는
               eligible 밖 시도의 거래도 포함한 상태로 계산되므로, 오염(다른 시도와 섞임)은
               정상적으로 감지된다.
    """
    ok = attempt >= 0
    comp, attempt = comp[ok], attempt[ok]
    if len(comp) == 0:
        return {'n_attempts': 0, 'n_scored': 0}
    ua, ai = np.unique(attempt, return_inverse=True)
    ub, bi = np.unique(comp, return_inverse=True)
    na, nb = len(ua), len(ub)

    pair = np.unique(ai.astype(np.int64) * nb + bi)          # (시도, 블록) 조합
    p_att, p_blk = pair // nb, pair % nb
    blocks_per_attempt = np.bincount(p_att, minlength=na)
    attempts_per_block = np.bincount(p_blk, minlength=nb)

    # 시도가 블록 하나에만 걸쳐 있고(쪼개짐 없음), 그 블록에 다른 시도가 없을 때만 strict.
    first = np.searchsorted(p_att, np.arange(na))             # pair 는 이미 시도 순 정렬
    blk_of = p_blk[np.minimum(first, len(pair) - 1)]
    strict = (blocks_per_attempt == 1) & (attempts_per_block[blk_of] == 1)

    score = np.ones(na, dtype=bool) if eligible is None else np.isin(ua, eligible)
    cnt = np.bincount(bi.astype(np.int64) * na + ai, minlength=nb * na).reshape(nb, na)
    tot = cnt.sum(1)
    purity = float((cnt.max(1) / np.maximum(tot, 1) * tot).sum() / tot.sum())
    extra = {'attempt_ids': ua, 'strict_flags': strict, 'scored': score} if per_attempt else {}
    return {**extra, 'n_attempts': int(na), 'n_scored': int(score.sum()),
            'n_blocks': int(nb),
            'strict_recovery': float(strict[score].mean()) if score.any() else float('nan'),
            'purity': purity,
            'frag_median': float(np.median(blocks_per_attempt[score])) if score.any() else float('nan'),
            'frag_mean': float(blocks_per_attempt[score].mean()) if score.any() else float('nan'),
            'mixed_blocks': int((attempts_per_block > 1).sum())}


## 5. HI-Large 클러스터/CSSMC 인덱스 생성 — `make_cluster_indices.py` (2026-09-10)

HI-Large에서 한 번도 못 돌았던 클러스터 기반 언더샘플링(정상 거래 1억 건 전량을 메모리에 올려야 해서
계속 죽었음)을 100만 행씩 청크로 처리하도록 고쳐 처음 실행. 같은 크기의 무작위 대조군(`random_r*`)과
비교하기 위해, 논문(김주미·정여진 2024) 기반 CSSMC(모집단과 KL 발산이 최소인 층화 무작위 subset 채택) 방식도 같이 만든다.

⚠️ 스크립트 docstring에 "논문에 없어서 내가 정한 것"이 명시돼 있음 — KL 계산식(피처별 히스토그램 평균),
후보 subset 수(N_CAND=20), 군집 피처공간(no_abs 76열)은 전부 에이전트 임의값. (참고: [[피처축소_사실확인_손은총_20260911.md]] §2)

원본 경로: `make_cluster_indices.py`

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""k-means 층화 추출 + CSSMC 언더샘플 인덱스 생성 — HI-Large train. 2026-09-10 (인계 0910 §8-4, 사용자 지시).

무엇을 만드나 (둘 다 **실제 행 선택**이다. 중심점 대체(논문 k-means*)가 아니다):
  cluster_r{r}_tr.npy — prep9/undersample.py 의 원래 동작. MiniBatchKMeans 군집 → 크기 비례 층화,
                        군집 안은 중심 근접 center_frac=0.5 + 무작위 0.5. HI-Large 에서 빠져 있던 이유는
                        메모리(정상 행 전량 적재 61 GB)였고, 여기서는 전량 배정을 청크로 한다(0909 §4.5.e).
  cssmc_r{r}_tr.npy   — 김주미·정여진 2024 (J. KIIT 22(5)) §3.1 (1)~(4):
                        (1) 다수클래스만 k-means → (2) 군집 기준 층화 무작위 추출로 subset 여러 개 →
                        (3) 모집단(train 다수클래스)과의 KL 발산이 가장 작은 subset 채택 → (4) 소수클래스와 결합.
왜 같이 만드나: 0910 §2-D — 같은 크기에서 CSSMC ≈ RUS 였는지(내 해석) 저자 주장(항상 우월)인지 우리 데이터로 확인.
    대조군은 이미 있는 random_r{r}_tr.npy (같은 keep_always + 같은 n_t) — 크기가 같아야 비교가 성립한다.

⚠️ 논문에 없어서 **내가 정한 것** (결과 json 에 그대로 기록):
  · KL 계산 방식 — 원문 수식(2)이 텍스트 추출에서 깨져 다차원 정의를 확인 못 함. 피처별 히스토그램 KL(P_모집단‖Q_subset)
    의 평균으로 계산. 연속 피처는 표본 분위수 NBIN 구간, 원핫(0/1) 피처는 2구간. 라플라스 평활 0.5.
  · 후보 subset 수 N_CAND — 원문에 수가 없음("여러 subset"). 후보별 KL 을 전부 기록해 선택이 투명하게.
  · 군집 피처 공간 — 모델이 보는 76열(no_abs). 원래 코드는 X_tr 81열 전부(절대시각 포함)를 썼다.
그대로 따른 것: SEED=42 · RATIOS=(10,30,100,300) · KMEANS_KS=(64,256,1024) · K 선택 "최적 대비 10% 이내 최소 K" ·
    fit 표본 100만 · sweep 표본 40만 · center_frac=0.5 (preprocess_9class.py:62-63, 250-255 · prep9/undersample.py 기본값).
    이 상수들은 팀 결정이 아니라 기존 파이프라인 값이다(감사_상수전수_20260909).
"""
import argparse, os, sys, json, time, gc
ap = argparse.ArgumentParser()
ap.add_argument('--dataset', default='HI-Large'); ap.add_argument('--basis', default='main')
ap.add_argument('--out', default=None, help='산출 폴더(기본: basis/undersample)')
ap.add_argument('--threads', type=int, default=2, help='스윕과 4코어를 나눠 쓴다')
ap.add_argument('--n-cand', type=int, default=20); ap.add_argument('--nbin', type=int, default=32)
ap.add_argument('--chunk', type=int, default=1_000_000)
ap.add_argument('--ratios', default='10,30,100,300')
a = ap.parse_args()
for v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[v] = str(a.threads)
import numpy as np, pandas as pd
from pathlib import Path
sys.path.insert(0, '/workspace')
from train9 import data as D
from prep9 import undersample as us
from sklearn.cluster import MiniBatchKMeans

SEED = 42; KMEANS_KS = (64, 256, 1024); FIT_SUB = 1_000_000; SWEEP_SUB = 400_000; CENTER_FRAC = 0.5
RATIOS = tuple(int(x) for x in a.ratios.split(','))
T0 = time.time()
def log(m): print(f'[{time.time()-T0:7.0f}s] {m}', flush=True)

b = D.load_basis(a.basis, dataset=a.dataset)
und = Path(a.out) if a.out else b.dir / 'undersample'
work = und / '_cluster_work'; work.mkdir(parents=True, exist_ok=True)
X = b.X('tr'); y = b.y('tr'); pos = b.is_pos('tr'); cols = b.cols('no_abs')
normal_tr = np.flatnonzero((y == 8) & ~pos)
keep_always = np.flatnonzero((y != 8) | pos)
n_pat = int((y <= 7).sum()); n_oop = int(((y == 8) & pos).sum()); N = len(normal_tr)
log(f'{a.dataset}/{a.basis} · train {len(y):,} · 정상 {N:,} · 패턴 {n_pat:,} · 패턴외양성 {n_oop:,} · 열 {len(cols)}')

# 1. 표본 100만 — 표준화 통계 · 구간 경계 · 중심 적합 (prep9 fit_subsample 과 같은 역할)
rng = np.random.default_rng(SEED)
sub = np.sort(rng.choice(N, min(FIT_SUB, N), replace=False))
Xs = np.asarray(X[normal_tr[sub]])[:, cols].astype(np.float32)
mean, std = Xs.mean(0), Xs.std(0)
edges = []
for j in range(Xs.shape[1]):
    u = np.unique(Xs[:, j])
    if len(u) <= 2 and set(u.tolist()) <= {0.0, 1.0}:
        edges.append(np.array([0.5], dtype=np.float32))
    else:
        q = np.unique(np.quantile(Xs[:, j], np.linspace(0, 1, a.nbin + 1)[1:-1]))
        edges.append(q.astype(np.float32))
n_bins = np.array([len(e) + 1 for e in edges]); assert n_bins.max() <= 255
log(f'표본 {len(sub):,}행 · 구간수 원핫 {int((n_bins==2).sum())}열 / 연속 {int((n_bins>2).sum())}열 (최대 {n_bins.max()})')

# 2. K 선택 — 기존 규칙 그대로 (preprocess_9class.py:250-253)
k_sweep = us.sweep_k(Xs, KMEANS_KS, seed=SEED, subsample=SWEEP_SUB)
best_err = min(r['inertia_per_row'] for r in k_sweep)
k_best = int(min(r['k'] for r in k_sweep if r['inertia_per_row'] <= best_err * 1.10))
pd.DataFrame(k_sweep).to_csv(und / 'kmeans_sweep_hi_large.csv', index=False)
log(f'K 선택 = {k_best} (규칙: 양자화 오차가 최적 대비 10% 이내인 가장 작은 K)')

# 3. 중심 적합 (prep9.fit_clusters 와 같은 파라미터)
Zs = ((Xs - mean) / (std + 1e-6)).astype(np.float32)
km = MiniBatchKMeans(n_clusters=k_best, random_state=SEED, batch_size=20_000, n_init=3, max_iter=100,
                     reassignment_ratio=0.01).fit(Zs)
C = km.cluster_centers_.astype(np.float32); del Zs, Xs; gc.collect()
log(f'중심 적합 완료 · inertia/row {km.inertia_/len(sub):.4f}')

# 4. 전량 한 번 훑기 — 군집 배정 · 중심 거리 · 히스토그램 구간 코드 · 모집단 히스토그램
lab = np.empty(N, dtype=np.int32); dist = np.empty(N, dtype=np.float32)
codes = np.lib.format.open_memmap(work / 'bin_codes_u8.npy', mode='w+', dtype=np.uint8, shape=(N, len(cols)))
pop = [np.zeros(nb, dtype=np.int64) for nb in n_bins]
for lo in range(0, N, a.chunk):
    hi = min(lo + a.chunk, N); rows = normal_tr[lo:hi]
    blk = np.asarray(X[rows[0]:rows[-1] + 1])[rows - rows[0]][:, cols]          # 연속 구간 읽고 정상 행만
    Z = ((blk - mean) / (std + 1e-6)).astype(np.float32)
    l = km.predict(Z); lab[lo:hi] = l
    dist[lo:hi] = np.sqrt(((Z - C[l]) ** 2).sum(1))
    cb = np.empty((hi - lo, len(cols)), dtype=np.uint8)
    for j in range(len(cols)):
        cj = np.searchsorted(edges[j], blk[:, j], side='right'); cb[:, j] = cj
        pop[j] += np.bincount(cj, minlength=n_bins[j])
    codes[lo:hi] = cb
    if (lo // a.chunk) % 10 == 0: log(f'  배정 {hi:,}/{N:,}')
codes.flush(); np.save(work / 'lab.npy', lab); np.save(work / 'dist.npy', dist)
sizes = np.bincount(lab, minlength=k_best)
log(f'배정 완료 · 빈 군집 {int((sizes==0).sum())} · 크기 {sizes.min()}~{sizes.max()}')

# 층화 구조 한 번만 만들어 재사용
order = np.argsort(lab, kind='stable'); offs = np.searchsorted(lab[order], np.arange(k_best + 1))
def quota_for(n_t):
    q = np.floor(sizes * (n_t / N)).astype(np.int64); short = n_t - int(q.sum())
    if short > 0:
        for c in np.argsort(sizes)[::-1][:short]: q[c] += 1
    return q
def stratified_random(n_t, seed):
    """CSSMC (2): 군집 비례 층화 **무작위** 추출 (center_frac=0 과 같음)."""
    r = np.random.default_rng(seed); q = quota_for(n_t); out = []
    for c in range(k_best):
        mem = order[offs[c]:offs[c + 1]]; qq = int(min(q[c], len(mem)))
        if qq: out.append(r.choice(mem, qq, replace=False))
    return np.sort(np.concatenate(out))
P = [p / p.sum() for p in pop]
def kl_of(pick):
    """KL(P_모집단 ‖ Q_subset), 피처별 평균. 라플라스 0.5."""
    cb = np.asarray(codes[pick]); tot = 0.0
    for j in range(len(cols)):
        qc = np.bincount(cb[:, j], minlength=n_bins[j]).astype(np.float64) + 0.5
        Q = qc / qc.sum(); m = P[j] > 0
        tot += float(np.sum(P[j][m] * np.log(P[j][m] / Q[m])))
    return tot / len(cols)

rep, cand_rows = [], []
for r in RATIOS:
    n_t = int(min(r * n_pat, N))
    # (a) k-means 층화 (원래 동작, center_frac=0.5)
    pick = us.sample_indices(lab, dist, n_t, center_frac=CENTER_FRAC, seed=SEED)
    idx = np.sort(np.concatenate([keep_always, normal_tr[pick]])); np.save(und / f'cluster_r{r}_tr.npy', idx.astype(np.int64))
    kl_c = kl_of(pick)
    rep.append(dict(method='cluster', ratio=r, n_normal=len(pick), n_total=len(idx), n_pattern=n_pat, n_oop_pos=n_oop,
                    clusters_covered=int(len(np.unique(lab[pick]))), kl=kl_c, center_frac=CENTER_FRAC))
    log(f'cluster_r{r}: 정상 {len(pick):,} · KL {kl_c:.6f}')
    # (b) CSSMC — 후보 N_CAND 개 중 KL 최소
    best = None
    for c in range(a.n_cand):
        pc = stratified_random(n_t, SEED * 1000 + c); k = kl_of(pc)
        cand_rows.append(dict(ratio=r, cand=c, seed=SEED * 1000 + c, kl=k))
        if best is None or k < best[0]: best = (k, c, pc)
    kl_b, c_b, pick = best
    idx = np.sort(np.concatenate([keep_always, normal_tr[pick]])); np.save(und / f'cssmc_r{r}_tr.npy', idx.astype(np.int64))
    kls = np.array([x['kl'] for x in cand_rows if x['ratio'] == r])
    rep.append(dict(method='cssmc', ratio=r, n_normal=len(pick), n_total=len(idx), n_pattern=n_pat, n_oop_pos=n_oop,
                    clusters_covered=int(len(np.unique(lab[pick]))), kl=kl_b, chosen_cand=c_b,
                    kl_cand_min=float(kls.min()), kl_cand_median=float(np.median(kls)), kl_cand_max=float(kls.max())))
    # (c) 참고 — 기존 random 팔의 KL (같은 잣대로)
    rp = und / f'random_r{r}_tr.npy'
    if rp.exists():
        ridx = np.load(rp); rpick = np.searchsorted(normal_tr, np.setdiff1d(ridx, keep_always, assume_unique=True))
        rep.append(dict(method='random(기존)', ratio=r, n_normal=len(rpick), n_total=len(ridx), n_pattern=n_pat,
                        n_oop_pos=n_oop, clusters_covered=int(len(np.unique(lab[rpick]))), kl=kl_of(rpick)))
    log(f'cssmc_r{r}: 후보 {a.n_cand} KL {kls.min():.6f}~{kls.max():.6f} → 후보 {c_b} 채택')

pd.DataFrame(rep).to_csv(und / 'variants_cluster.csv', index=False)
pd.DataFrame(cand_rows).to_csv(und / 'cssmc_candidates.csv', index=False)
json.dump(dict(dataset=a.dataset, basis=a.basis, seed=SEED, k=k_best, k_sweep=k_sweep, fit_sub=len(sub),
               n_cand=a.n_cand, nbin=a.nbin, cols=int(len(cols)), feature_set='no_abs', center_frac=CENTER_FRAC,
               my_choices=['KL=피처별 히스토그램 KL 평균(라플라스 0.5)', f'N_CAND={a.n_cand}', f'NBIN={a.nbin}', '군집 피처공간=no_abs 76열'],
               inherited=['SEED 42', 'RATIOS', 'KMEANS_KS (64,256,1024)', 'K 10% 규칙', 'fit 100만/sweep 40만', 'center_frac 0.5'],
               cluster_sizes_min=int(sizes.min()), cluster_sizes_max=int(sizes.max()), elapsed_s=round(time.time() - T0, 1)),
          open(und / 'cluster_meta.json', 'w'), ensure_ascii=False, indent=1)
print(pd.DataFrame(rep).to_string(index=False)); log('CLUSTER_INDICES_DONE')


## 6. 언더샘플 비율 격자 확장 — `make_ratio_indices.py` (2026-09-10)

기존 r10/r30/r100/r300 4벌에 r350/r400/r450/r500을 추가. 단순히 새 난수로 뽑으면 기존 4벌과
다른 계열이 되므로, 같은 `np.random.default_rng(SEED)` 흐름에서 기존 draw를 먼저 재생해
바이트 단위로 일치하는지 검증한 뒤 그 흐름을 이어서 새 비율을 뽑는다(불일치 시 아무것도 안 쓰고 중단).

원본 경로: `make_ratio_indices.py`

In [ ]:
# -*- coding: utf-8 -*-
"""r350/400/450/500 언더샘플 인덱스 생성 — 기존 4벌과 같은 난수 흐름을 이어서 뽑는다.

왜 이렇게 하나:
  preprocess_9class.py:259 가 `rng = np.random.default_rng(SEED)` 를 한 번 만들고
  UNDERSAMPLE_RATIOS 를 순서대로 돌며 draw 를 소비한다. 그래서 새 비율을 그냥 새 rng 로 뽑으면
  기존 4벌과 다른 계열이 된다. 여기서는 10/30/100/300 draw 를 먼저 **똑같이 재생**해
  기존 파일과 바이트 단위로 일치하는지 검증한 뒤, 그 흐름을 이어서 350~500 을 뽑는다.
  일치하지 않으면 아무것도 쓰지 않고 중단한다.
"""
import os, sys
for v in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'):
    os.environ[v] = '3'
import numpy as np
from pathlib import Path
from train9 import data as D

SEED = 42                       # preprocess_9class.py:34
CLASS8 = 8
OLD = (10, 30, 100, 300)        # preprocess_9class.py:62
NEW = (350, 400, 450, 500)

b = D.load_basis('main', dataset='HI-Large')
und = b.dir / 'undersample'
y9tr = b.y('tr'); postr = b.is_pos('tr')
normal_tr   = np.flatnonzero((y9tr == CLASS8) & (~postr))
keep_always = np.flatnonzero((y9tr != CLASS8) | postr)
n_pattern_tr = int((y9tr <= 7).sum())
print(f'train 정상 {len(normal_tr):,} · 패턴 {n_pattern_tr:,} · 항상유지 {len(keep_always):,}', flush=True)

rng = np.random.default_rng(SEED)

def draw(r):
    n_t = min(r * n_pattern_tr, len(normal_tr))
    pick = normal_tr[np.sort(rng.choice(len(normal_tr), n_t, replace=False))]
    return np.sort(np.concatenate([keep_always, pick])).astype(np.int64)

# 1) 기존 4벌 재생 + 대조
for r in OLD:
    got = draw(r)
    ref = np.load(und / f'random_r{r}_tr.npy')
    ok = got.shape == ref.shape and np.array_equal(got, ref)
    print(f'  검증 r{r:<4d} {got.shape[0]:>10,}행 · 기존과 일치: {ok}', flush=True)
    if not ok:
        sys.exit(f'중단 — r{r} 재현 실패. 난수 흐름이 다르므로 새 인덱스를 만들지 않는다.')

# 2) 흐름을 이어서 새 비율
print('재현 검증 통과 — 새 비율 생성', flush=True)
for r in NEW:
    idx = draw(r)
    p = und / f'random_r{r}_tr.npy'
    np.save(p, idx)
    print(f'  생성 r{r:<4d} {len(idx):>10,}행 (정상 {len(idx)-len(keep_always):>10,}) → {p.name} '
          f'{p.stat().st_size/1024**2:.0f}MB', flush=True)
print('완료', flush=True)


## 부록 — 이번에 전문을 안 실은 관련 모듈

| 파일 | 줄 수 | 역할 |
|---|---:|---|
| `prep9/lowmem.py` | 563 | cgroup 27.3GiB 상한 안에서 HI-Large(1.8억 행)를 처리하기 위한 저메모리 전처리 함수 모음(2026-09-02). `preprocess_9class.py --lowmem`이 원본 함수를 값 동일한 메모리 절약판으로 갈아끼울 때 씀. HI-Small로 원본과 동치 검증됨(`compare_lowmem_small.py`). |
| `prep9/coords.py` | 49 | 좌표계 변환 유틸(회전 인덱스 등, 블록 시각화 보조) |
| `prep9/ego.py` | 164 | 블록(전역 분할) 대신 알림마다 국소 서브그래프를 쓰는 대안 구현 — 채택 안 됨 |
| `prep9/make_readme.py`, `prep9/md2docx.py` | 107 / 174 | 산출물 문서·보고서 자동 생성 유틸(전처리 로직 아님) |

전문이 필요하면 말씀해주세요 — 이 노트북에 추가하거나 따로 만들 수 있어요.
